# 01 – Data Collection

This notebook downloads historical OHLCV data for a list of stock tickers using **yfinance** (free, no API key required) and optionally **Alpha Vantage** (free API key at [alphavantage.co](https://www.alphavantage.co)).

**Output:** raw CSV files saved to `data/raw/<TICKER>.csv`

In [ ]:
# ── Imports & path setup ──────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from src.data_collection import StockDataCollector

print('✅ Imports OK')

## 1.1 Define tickers and date range

In [ ]:
TICKERS    = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']
START_DATE = '2020-01-01'
END_DATE   = '2024-12-31'
INTERVAL   = '1d'          # '1d', '1wk', '1mo'

print(f'Tickers : {TICKERS}')
print(f'Period  : {START_DATE} → {END_DATE}')

## 1.2 Download via yfinance

In [ ]:
collector = StockDataCollector(tickers=TICKERS)
data = collector.fetch_all(start=START_DATE, end=END_DATE, interval=INTERVAL, save=True)

print(f'\n📦 Downloaded {len(data)} ticker(s):')
for ticker, df in data.items():
    print(f'  {ticker:6s} → {df.shape[0]:>5d} rows | {df.index.min().date()} → {df.index.max().date()}')

## 1.3 Quick preview

In [ ]:
aapl = data['AAPL']
print(f'Shape: {aapl.shape}')
aapl.head()

In [ ]:
aapl.describe().round(2)

## 1.4 Check for missing values across all tickers

In [ ]:
missing_report = {}
for ticker, df in data.items():
    missing_report[ticker] = df.isnull().sum().to_dict()

pd.DataFrame(missing_report).T

## 1.5 Closing price overview plot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

fig, ax = plt.subplots(figsize=(14, 6))

for ticker, df in data.items():
    ax.plot(df.index, df['close'], label=ticker, linewidth=1.4)

ax.set_title('Closing Prices – All Tickers', fontsize=15, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Price (USD)')
ax.legend(loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig('../images/charts/all_tickers_close.png', dpi=150)
plt.show()

## 1.6 (Optional) Alpha Vantage

In [ ]:
# Uncomment and add your API key to fetch from Alpha Vantage
# from src.data_collection import fetch_alpha_vantage
# import os
# 
# API_KEY = os.getenv('ALPHA_VANTAGE_API_KEY', 'YOUR_KEY_HERE')
# av_df = fetch_alpha_vantage('AAPL', api_key=API_KEY)
# av_df.tail()

---
**Next →** `02_data_cleaning.ipynb`